In [10]:
import sys
import polars as pl
from pathlib import Path
import json

ROOT = Path().resolve()

# поднимаемся до проекта (а не src!)
while not (ROOT / "src").exists():
    ROOT = ROOT.parent

sys.path.append(str(ROOT / "src"))

from ingestion import extract_imdb

In [11]:
imdb_path = ROOT / "data/raw/imdb/basics.tsv"

print(imdb_path)
print(imdb_path.exists())

C:\Users\Utilisateur\Simplon\Simplon-selenium\data\raw\imdb\basics.tsv
True


In [13]:
df_raw = extract_imdb(limit=100)


basics already downloaded
principals already downloaded
names already downloaded
IMDb download completed.
Unzipping basics.tsv.gz
Unzipping names.tsv.gz
Unzipping principals.tsv.gz


In [14]:
len(df_raw)

100

In [15]:
df_raw.select("genres").unique()

genres
str
"""News,Short,Sport"""
"""Animation,Short"""
"""Documentary,Short"""
"""Horror,Short"""
"""News,Short"""
…
"""Animation,Comedy,Short"""
"""Animation,Comedy,Romance"""
"""Romance"""


In [ ]:
def normalize_imdb(df: pl.DataFrame) -> pl.DataFrame:
    return (
        df
        .with_columns([
            # --- STRINGS CLEAN ---
            pl.col("title").fill_null("").str.strip_chars(),
            pl.col("genres").fill_null("").str.strip_chars(),

            # --- YEAR ---
            pl.col("year")
              .cast(pl.Int32, strict=False),

            # --- CAST FIX ---
            pl.col("cast").fill_null([]),

            # --- NORMALIZED TYPES ---
            pl.col("imdb_id").cast(pl.Utf8),
        ])
    )

In [ ]:
df_clean = normalize_imdb(df_raw)
df_clean

imdb_id,title,year,genres,cast
str,str,i32,str,list[str]
"""tt0000001""","""Carmencita""",1894,"""Documentary,Short""",[]
"""tt0000002""","""Le clown et ses chiens""",1892,"""Animation,Short""",[]
"""tt0000003""","""Poor Pierrot""",1892,"""Animation,Comedy,Romance""",[]
"""tt0000004""","""Un bon bock""",1892,"""Animation,Short""",[]
"""tt0000005""","""Blacksmith Scene""",1893,"""Short""","[""Charles Kayser"", ""John Ott""]"
…,…,…,…,…
"""tt0010138""","""For Bitter or for Verse""",1919,"""Animation,Comedy,Short""",[]
"""tt0010139""","""For a Woman's Honor""",1919,"""Drama""","[""H.B. Warner"", ""Marguerite De La Motte"", … ""Hector V. Sarno""]"
"""tt0010140""","""Forbidden""",1919,"""Drama""","[""Mildred Harris"", ""Henry Woodward"", … ""Harry Woodward""]"
